# Morpho-Algo — SFT Training (Phi-4-mini 3.8B)

Trains the trading-decision model via full-parameter bf16 SFT on P1 supervised samples.

**Runtime required:** A **GPU** runtime. Recommended: `A100` (40GB) under Runtime → Change runtime type → Hardware accelerator: **GPU** (T4 also works but is slower).

Everything (data + fixed driver) is pulled from Hugging Face `bluemorpholimited/morpho-algo`. Checkpoints are **uploaded back to HF** every 300 steps and the run **auto-resumes** from the latest checkpoint if it ever interrupts.

## 0. Set your Hugging Face token

Enter your **bluemorpholimited** write token (NOT the pink/dragon tokens). It must have write access to `bluemorpholimited/morpho-algo`. It is stored only in this Colab session's environment, not in the repo.

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your bluemorpholimited HF write token: ")
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN length:", len(HF_TOKEN), "| set OK")

## 1. Install dependencies

In [ ]:
%pip install -q torch transformers accelerate huggingface_hub numpy

## 2. Pull the fixed SFT driver + data from HF

In [ ]:
import os, urllib.request, shutil

REPO = "bluemorpholimited/morpho-algo"
WORK = "/content/morpho_algo"
CKPT = f"{WORK}/checkpoints/sft"
SPLITS = f"{WORK}/p1_splits"
os.makedirs(CKPT, exist_ok=True)
os.makedirs(SPLITS, exist_ok=True)

HF = os.environ["HF_TOKEN"]

# fixed driver (true resume + real HF ckpt upload)
url = f"https://huggingface.co/{REPO}/resolve/main/src/morpho_sft.py"
req = urllib.request.Request(url, headers={"Authorization": f"Bearer {HF}"})
open(f"{WORK}/morpho_sft.py", "wb").write(urllib.request.urlopen(req).read())
print("driver ->", f"{WORK}/morpho_sft.py")

from huggingface_hub import snapshot_download
for sub in ("datasets/P1_v1/splits/train", "datasets/P1_v1/splits/valid"):
    snapshot_download(repo_id=REPO, repo_type="model", token=HF,
                      allow_patterns=[f"{sub}/**"], local_dir=SPLITS)
print("data ->", SPLITS)

## 3. Run SFT

Runs full-parameter bf16 on the P1 train+valid splits. Every 300 optimizer steps a checkpoint is saved **and uploaded to HF**. If this cell is ever interrupted, just re-run it — `--resume auto` reloads the latest checkpoint's real weights and continues.

In [ ]:
import os, subprocess, sys, torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Enable Runtime > Change runtime type > Hardware accelerator = GPU, then rerun.')

train = f"{SPLITS}/datasets/P1_v1/splits/train"
valid = f"{SPLITS}/datasets/P1_v1/splits/valid"
assert os.path.isdir(train), f"missing {train}"

cmd = [sys.executable, f"{WORK}/morpho_sft.py",
       "--data-train", train, "--data-valid", valid,
       "--out", CKPT, "--model", "microsoft/Phi-4-mini-instruct",
       "--epochs", "3", "--bs", "2", "--grad-accum", "8",
       "--max-len", "900", "--save-every", "300", "--log-every", "10",
       "--run-tag", "sft-v1", "--repo", REPO, "--resume", "auto"]
print("Launching SFT...")
p = subprocess.run(cmd)
print("SFT exit", p.returncode)

## After training

Final model + every `step-*` checkpoint are on HF under `bluemorpholimited/morpho-algo/checkpoints/sft/sft-v1/`. Tell the assistant once training completes so the **DPO** stage can be staged next.